# 13 · Classifieurs — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch13_classifieurs/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch13_classifieurs/03_notebook.ipynb`, créée par `python tools/start_chapter.py 13`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 13.Q1 à 13.Q12, 13.R1 à 13.R3, 13.1 à 13.7, 13.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★★ | |
| A | 13.12 à 13.15 | prédire les frontières de cinq classifieurs, choisir k, KNeighborsClassifier dans mylearn.neighbors, kNN et malédiction de la dimension (*prochaine session de génération*) | | | |
| B | 13.16 à 13.18 | SVM linéaire et paramètre C, relever des cercles en 3D, noyau RBF réglé par validation croisée (*prochaine session de génération*) | | | |
| C | 13.19 à 13.26 | instabilité d'un arbre, impuretés, meilleur split et arbres dans mylearn.tree, une colonne qui triche, plot_tree et élagage, arbre de régression (*prochaine session de génération*) | | | |
| D | 13.27 à 13.30 | Naïve Bayes sur des blobs puis des lunes, GaussianNB et MultinomialNB dans mylearn.naive_bayes, phrases de Holmes en sac de mots et TF-IDF (*prochaine session de génération*) | | | |
| E | 13.31 et 13.32 | régression logistique de scikit-learn, puis la tienne dans mylearn.logistic (*prochaine session de génération*) | | | |
| F | 13.33 à 13.36 | associer des frontières, accélérer son kNN, comparatif mylearn contre scikit-learn, défi MNIST (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")  # your own library

## Objectifs et rappel express

- Calculer à la main un vote de k plus proches voisins, une marge, une impureté, un gain de split et un posterior de Naïve Bayes.
- Écrire et vérifier contre scikit-learn les classifieurs de ta librairie : `KNeighborsClassifier`, les arbres de `mylearn.tree` (prêts pour les ensembles du ch. 14), `GaussianNB`, `MultinomialNB` et `LogisticRegression`.
- Régler $k$, C, $\gamma$ et la profondeur d'un arbre par validation croisée, et limiter l'overfitting d'un arbre.
- Comparer les classifieurs (accuracy, temps, mémoire, explicabilité) et choisir une baseline.

**Rappel express.** Distances : $\sqrt{\sum_j (x_j - z_j)^2}$ et $\sum_j \lvert x_j - z_j \rvert$ ; distance à un hyperplan : $\lvert \mathbf{w} \cdot \mathbf{x} + b \rvert / \lVert \mathbf{w} \rVert$, largeur de la bande $2/\lVert \mathbf{w} \rVert$ ; noyau RBF : $e^{-\gamma \lVert \mathbf{x} - \mathbf{z} \rVert^2}$ ; Gini : $1 - \sum_c p_c^2$ ; entropie : $-\sum_c p_c \log_2 p_c$ ; gain : $I(\text{parent}) - \frac{n_G}{n} I(G) - \frac{n_D}{n} I(D)$ ; Naïve Bayes : $\log P(c) + \sum_j \log P(x_j \mid c)$, normalisé par logsumexp ; lissage : $\frac{N_{cj} + \alpha}{N_c + \alpha d}$ ; régression logistique : $\hat p = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$, gradient $\frac1n \mathbf{X}^\top (\hat{\mathbf{p}} - \mathbf{y})$. En Python : `sklearn.neighbors.KNeighborsClassifier`, `sklearn.svm.SVC`, `sklearn.tree` (`DecisionTreeClassifier`, `plot_tree`, `export_text`), `sklearn.naive_bayes` (`GaussianNB`, `MultinomialNB`), `sklearn.linear_model.LogisticRegression`, `sklearn.model_selection.GridSearchCV`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 13.1 à 13.7 et 🧮 13.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une suite de codes : les lettres dans l'ordre (`"XYZ"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 13.8, la réflexion (🗣️ 13.10, ⚖️ 13.11) et l'entretien se corrigent avec `05_solutions.md`.

**13.Q1 — Pourquoi essayer des classifieurs classiques d'abord ?**

In [ ]:
# 13.Q1: the values you found on paper
answer_13_Q1a = ...  # a) True or False
answer_13_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_13_Q1c = ...  # c) True or False
answer_13_Q1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_13_Q1a, answer_13_Q1b, answer_13_Q1c, answer_13_Q1d]):
    wb.check(f"13.Q1{letter}", answer)

**13.Q2 — Paramétrique ou non paramétrique ?**

In [ ]:
# 13.Q2: the values you found on paper
answer_13_Q2a = ...  # a) six letters P or N, in the order of the items, e.g. "XYXYXY"
answer_13_Q2b = ...  # b) the letter of your choice
answer_13_Q2c = ...  # c) True or False
answer_13_Q2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_13_Q2a, answer_13_Q2b, answer_13_Q2c, answer_13_Q2d]):
    wb.check(f"13.Q2{letter}", answer)

**13.Q3 — kNN n'est pas k-means**

In [ ]:
# 13.Q3: the values you found on paper
answer_13_Q3a = ...  # a) the letter of your choice
answer_13_Q3b = ...  # b) the letter of your choice
answer_13_Q3c = ...  # c) True or False
answer_13_Q3d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_13_Q3a, answer_13_Q3b, answer_13_Q3c, answer_13_Q3d]):
    wb.check(f"13.Q3{letter}", answer)

**13.Q4 — Petit k, grand k**

In [ ]:
# 13.Q4: the values you found on paper
answer_13_Q4a = ...  # a) a number
answer_13_Q4b = ...  # b) the letter of your choice
answer_13_Q4c = ...  # c) the letter of your choice
answer_13_Q4d = ...  # d) True or False
answer_13_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q4a, answer_13_Q4b, answer_13_Q4c, answer_13_Q4d, answer_13_Q4e]):
    wb.check(f"13.Q4{letter}", answer)

**13.Q5 — Marge, vecteurs de support et C**

In [ ]:
# 13.Q5: the values you found on paper
answer_13_Q5a = ...  # a) True or False
answer_13_Q5b = ...  # b) the letter of your choice
answer_13_Q5c = ...  # c) the letter of your choice
answer_13_Q5d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_13_Q5a, answer_13_Q5b, answer_13_Q5c, answer_13_Q5d]):
    wb.check(f"13.Q5{letter}", answer)

**13.Q6 — Ce que l'astuce du noyau évite de calculer**

In [ ]:
# 13.Q6: the values you found on paper
answer_13_Q6a = ...  # a) the letter of your choice
answer_13_Q6b = ...  # b) a whole number
answer_13_Q6c = ...  # c) a number
answer_13_Q6d = ...  # d) True or False
answer_13_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q6a, answer_13_Q6b, answer_13_Q6c, answer_13_Q6d, answer_13_Q6e]):
    wb.check(f"13.Q6{letter}", answer)

**13.Q7 — Racine, feuilles, profondeur : le vocabulaire des arbres**

In [ ]:
# 13.Q7: the values you found on paper
answer_13_Q7a = ...  # a) a whole number
answer_13_Q7b = ...  # b) a whole number
answer_13_Q7c = ...  # c) a whole number
answer_13_Q7d = ...  # d) a whole number
answer_13_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q7a, answer_13_Q7b, answer_13_Q7c, answer_13_Q7d, answer_13_Q7e]):
    wb.check(f"13.Q7{letter}", answer)

**13.Q8 — Pourquoi un arbre surapprend-il ?**

In [ ]:
# 13.Q8: the values you found on paper
answer_13_Q8a = ...  # a) a number
answer_13_Q8b = ...  # b) True or False
answer_13_Q8c = ...  # c) the letter of your choice
answer_13_Q8d = ...  # d) the letter of your choice

for letter, answer in zip("abcd", [answer_13_Q8a, answer_13_Q8b, answer_13_Q8c, answer_13_Q8d]):
    wb.check(f"13.Q8{letter}", answer)

**13.Q9 — Gini ou gain d'information ?**

In [ ]:
# 13.Q9: the values you found on paper
answer_13_Q9a = ...  # a) a number
answer_13_Q9b = ...  # b) a number
answer_13_Q9c = ...  # c) a number
answer_13_Q9d = ...  # d) the letter of your choice
answer_13_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q9a, answer_13_Q9b, answer_13_Q9c, answer_13_Q9d, answer_13_Q9e]):
    wb.check(f"13.Q9{letter}", answer)

**13.Q10 — Limiter la profondeur ou élaguer ?**

In [ ]:
# 13.Q10: the values you found on paper
answer_13_Q10a = ...  # a) five letters P or E, in the order of the items, e.g. "XYXYX"
answer_13_Q10b = ...  # b) True or False
answer_13_Q10c = ...  # c) the letter of your choice

for letter, answer in zip("abc", [answer_13_Q10a, answer_13_Q10b, answer_13_Q10c]):
    wb.check(f"13.Q10{letter}", answer)

**13.Q11 — Ce qui est vraiment « naïf » dans Naïve Bayes**

In [ ]:
# 13.Q11: the values you found on paper
answer_13_Q11a = ...  # a) the letter of your choice
answer_13_Q11b = ...  # b) True or False
answer_13_Q11c = ...  # c) a whole number
answer_13_Q11d = ...  # d) True or False
answer_13_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q11a, answer_13_Q11b, answer_13_Q11c, answer_13_Q11d, answer_13_Q11e]):
    wb.check(f"13.Q11{letter}", answer)

**13.Q12 — Quel classifieur pour quelle situation ?**

In [ ]:
# 13.Q12: the values you found on paper
answer_13_Q12a = ...  # a) five letters A, N, K, L, S, in the order of the situations, e.g. "AAAAA"
answer_13_Q12b = ...  # b) True or False

for letter, answer in zip("ab", [answer_13_Q12a, answer_13_Q12b]):
    wb.check(f"13.Q12{letter}", answer)

**13.R1 — Pourquoi standardiser avant de mesurer des distances**

In [ ]:
# 13.R1: the values you found on paper
answer_13_R1a = ...  # a) a number with 2 decimals
answer_13_R1b = ...  # b) a number with 2 decimals
answer_13_R1c = ...  # c) the letter "A" or "B"
answer_13_R1d = ...  # d) a number with 3 decimals
answer_13_R1e = ...  # e) the letter "A" or "B"
answer_13_R1f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_13_R1a, answer_13_R1b, answer_13_R1c, answer_13_R1d, answer_13_R1e, answer_13_R1f]):
    wb.check(f"13.R1{letter}", answer)

**13.R2 — L'équation de la frontière d'un perceptron**

In [ ]:
# 13.R2: the values you found on paper
answer_13_R2a = ...  # a) 0 or 1
answer_13_R2b = ...  # b) 0 or 1
answer_13_R2c = ...  # c) the letter of your choice
answer_13_R2d = ...  # d) a number
answer_13_R2e = ...  # e) a number
answer_13_R2f = ...  # f) a number with 3 decimals
answer_13_R2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_13_R2a, answer_13_R2b, answer_13_R2c, answer_13_R2d, answer_13_R2e, answer_13_R2f, answer_13_R2g]):
    wb.check(f"13.R2{letter}", answer)

**13.R3 — Entropie de deux distributions**

In [ ]:
# 13.R3: the values you found on paper
answer_13_R3a = ...  # a) a number
answer_13_R3b = ...  # b) a number with 3 decimals
answer_13_R3c = ...  # c) a number
answer_13_R3d = ...  # d) a number with 3 decimals
answer_13_R3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_R3a, answer_13_R3b, answer_13_R3c, answer_13_R3d, answer_13_R3e]):
    wb.check(f"13.R3{letter}", answer)

**Ex 13.1 — Vote des k plus proches voisins à la main**

In [ ]:
# Ex 13.1: the values you found on paper
answer_13_1a = ...  # a) an index from 0 to 6
answer_13_1b = ...  # b) a class: 0, 1 or 2
answer_13_1c = ...  # c) a class: 0, 1 or 2
answer_13_1d = ...  # d) a number with 3 decimals
answer_13_1e = ...  # e) a class: 0, 1 or 2
answer_13_1f = ...  # f) a class: 0, 1 or 2
answer_13_1g = ...  # g) a number with 3 decimals
answer_13_1h = ...  # h) a class: 0, 1 or 2
answer_13_1i = ...  # i) a list [index, class]

for letter, answer in zip("abcdefghi", [answer_13_1a, answer_13_1b, answer_13_1c, answer_13_1d, answer_13_1e, answer_13_1f, answer_13_1g, answer_13_1h, answer_13_1i]):
    wb.check(f"13.1{letter}", answer)

**Ex 13.2 — Gini et entropie d'une feuille**

In [ ]:
# Ex 13.2: the values you found on paper
answer_13_2a = ...  # a) a number
answer_13_2b = ...  # b) a number with 3 decimals
answer_13_2c = ...  # c) a number
answer_13_2d = ...  # d) a number
answer_13_2e = ...  # e) a number with 3 decimals
answer_13_2f = ...  # f) a number
answer_13_2g = ...  # g) a number with 3 decimals
answer_13_2h = ...  # h) a number with 3 decimals
answer_13_2i = ...  # i) a number with 3 decimals

for letter, answer in zip("abcdefghi", [answer_13_2a, answer_13_2b, answer_13_2c, answer_13_2d, answer_13_2e, answer_13_2f, answer_13_2g, answer_13_2h, answer_13_2i]):
    wb.check(f"13.2{letter}", answer)

**Ex 13.3 — Choisir le meilleur seuil sur la longueur des nageoires**

In [ ]:
# Ex 13.3: the values you found on paper
answer_13_3a = ...  # a) a whole number
answer_13_3b = ...  # b) a number
answer_13_3c = ...  # c) a number
answer_13_3d = ...  # d) a number with 3 decimals
answer_13_3e = ...  # e) a number with 3 decimals
answer_13_3f = ...  # f) a threshold in mm
answer_13_3g = ...  # g) a number with 3 decimals
answer_13_3h = ...  # h) a whole number
answer_13_3i = ...  # i) 0 (Adélie) or 1 (Papou)
answer_13_3j = ...  # j) a whole number

for letter, answer in zip("abcdefghij", [answer_13_3a, answer_13_3b, answer_13_3c, answer_13_3d, answer_13_3e, answer_13_3f, answer_13_3g, answer_13_3h, answer_13_3i, answer_13_3j]):
    wb.check(f"13.3{letter}", answer)

**Ex 13.4 — Distance à une droite et marge d'un SVM**

In [ ]:
# Ex 13.4: the values you found on paper
answer_13_4a = ...  # a) a number with 4 decimals
answer_13_4b = ...  # b) a number
answer_13_4c = ...  # c) a number with 3 decimals
answer_13_4d = ...  # d) a number
answer_13_4e = ...  # e) a list of indices, in increasing order
answer_13_4f = ...  # f) a number with 3 decimals
answer_13_4g = ...  # g) +1 or -1
answer_13_4h = ...  # h) True or False
answer_13_4i = ...  # i) True or False
answer_13_4j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_13_4a, answer_13_4b, answer_13_4c, answer_13_4d, answer_13_4e, answer_13_4f, answer_13_4g, answer_13_4h, answer_13_4i, answer_13_4j]):
    wb.check(f"13.4{letter}", answer)

**Ex 13.5 — L'astuce du noyau vérifiée à la main**

In [ ]:
# Ex 13.5: the values you found on paper
answer_13_5a = ...  # a) a number
answer_13_5b = ...  # b) a number
answer_13_5c = ...  # c) a number with 3 decimals
answer_13_5d = ...  # d) a number
answer_13_5e = ...  # e) a number
answer_13_5f = ...  # f) a number with 4 decimals
answer_13_5g = ...  # g) a number with 2 decimals
answer_13_5h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_13_5a, answer_13_5b, answer_13_5c, answer_13_5d, answer_13_5e, answer_13_5f, answer_13_5g, answer_13_5h]):
    wb.check(f"13.5{letter}", answer)

**Ex 13.6 — Anglais ou français ? Naïve Bayes multinomial sur des lettres**

In [ ]:
# Ex 13.6: the values you found on paper
answer_13_6a = ...  # a) a number
answer_13_6b = ...  # b) a number with 3 decimals
answer_13_6c = ...  # c) a number with 4 decimals
answer_13_6d = ...  # d) a number with 4 decimals
answer_13_6e = ...  # e) a number with 3 decimals
answer_13_6f = ...  # f) a number with 3 decimals
answer_13_6g = ...  # g) a number with 3 decimals
answer_13_6h = ...  # h) a number
answer_13_6i = ...  # i) a number with 3 decimals

for letter, answer in zip("abcdefghi", [answer_13_6a, answer_13_6b, answer_13_6c, answer_13_6d, answer_13_6e, answer_13_6f, answer_13_6g, answer_13_6h, answer_13_6i]):
    wb.check(f"13.6{letter}", answer)

**Ex 13.7 — Posterior d'un Naïve Bayes gaussien**

In [ ]:
# Ex 13.7: the values you found on paper
answer_13_7a = ...  # a) a number
answer_13_7b = ...  # b) a number
answer_13_7c = ...  # c) a number with 3 decimals
answer_13_7d = ...  # d) a number with 3 decimals
answer_13_7e = ...  # e) a number with 3 decimals
answer_13_7f = ...  # f) a number with 3 decimals
answer_13_7g = ...  # g) a number with 3 decimals
answer_13_7h = ...  # h) a number with 3 decimals
answer_13_7i = ...  # i) a number with 3 decimals
answer_13_7j = ...  # j) a number with 3 decimals
answer_13_7k = ...  # k) True or False

for letter, answer in zip("abcdefghijk", [answer_13_7a, answer_13_7b, answer_13_7c, answer_13_7d, answer_13_7e, answer_13_7f, answer_13_7g, answer_13_7h, answer_13_7i, answer_13_7j, answer_13_7k]):
    wb.check(f"13.7{letter}", answer)

**Ex 13.9 — Mémoire et temps d'un kNN sur MNIST complet**

In [ ]:
# Ex 13.9: the values you found on paper
answer_13_9a = ...  # a) a whole number
answer_13_9b = ...  # b) a number with 2 decimals (MB)
answer_13_9c = ...  # c) a number with 2 decimals (MB)
answer_13_9d = ...  # d) a whole number
answer_13_9e = ...  # e) a number with 2 decimals (seconds)
answer_13_9f = ...  # f) a number (GB)
answer_13_9g = ...  # g) a number (MB)
answer_13_9h = ...  # h) a whole number
answer_13_9i = ...  # i) a whole number (rounded)
answer_13_9j = ...  # j) a number (MB)

for letter, answer in zip("abcdefghij", [answer_13_9a, answer_13_9b, answer_13_9c, answer_13_9d, answer_13_9e, answer_13_9f, answer_13_9g, answer_13_9h, answer_13_9i, answer_13_9j]):
    wb.check(f"13.9{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire, pour chaque classifieur du chapitre, ce qu'il garde en mémoire, où il passe son temps et ce qu'il suppose des données ?
2. Sais-tu calculer à la main un vote kNN, une marge, un gain de split et un posterior de Naïve Bayes ?
3. Sais-tu choisir une baseline et régler ses hyperparamètres par validation croisée ?

**Pour aller plus loin** : les guides utilisateur de scikit-learn sur les plus proches voisins, les SVM, les arbres et Naïve Bayes, et l'article de Domingos et Pazzani, cités dans la fiche. La suite : le ch. 14 (ensembles), qui construira forêts aléatoires, AdaBoost et gradient boosting avec tes arbres de `mylearn.tree`.

*Les exercices de code 13.12 à 13.36 (parties A à F, ta librairie `neighbors`, `tree`, `naive_bayes` et `logistic`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 13` pour obtenir le notebook complet.*